# MoCo v3 (ViT-S/16) Pretraining — Google Colab (L4)

## 사전 준비 (최초 1회)
1. Google Drive `내 드라이브/ssl_project/` 폴더에 프로젝트 파일 업로드
   - 업로드 대상: `ssl_lib/`, `scripts/`, `configs/`, `requirements.txt`, `setup.py`
   - `data/`, `outputs/`, `logs/` 는 업로드 불필요 (자동 생성)
2. Colab 런타임 유형: **L4 GPU** 선택

## 세션 재시작 시
Cell 1~3 다시 실행 → Cell 4가 마지막 체크포인트부터 자동으로 이어서 시작

## 파일 저장 위치
- 코드 실행: `/content/ssl_project/` (Colab 로컬 — 빠른 SSD)
- 체크포인트 / 로그: `내 드라이브/ssl_project/outputs|logs/` (Drive — 영구 보관)

In [ ]:
# Cell 1 — GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU가 없습니다. 런타임 → 런타임 유형 변경 → L4 GPU 선택'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2 — Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Drive에서 코드 복사 + 패키지 설치 + 심링크
import os, subprocess, shutil

DRIVE_DIR = '/content/drive/MyDrive/ssl_project'
WORK_DIR  = '/content/ssl_project'

# Drive → /content/ 로 코드 복사 (Drive 직접 실행은 I/O 느림)
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)

for item in ['ssl_lib', 'scripts', 'configs', 'requirements.txt', 'setup.py']:
    src = f'{DRIVE_DIR}/{item}'
    dst = f'{WORK_DIR}/{item}'
    if os.path.isdir(src):
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f'  copied: {item}')

os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')

# 패키지 설치
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)

# data / outputs / logs → Drive 심링크
for d in ['data', 'outputs', 'logs']:
    remote = f'{DRIVE_DIR}/{d}'
    local  = f'{WORK_DIR}/{d}'
    os.makedirs(remote, exist_ok=True)
    if os.path.exists(local) or os.path.islink(local):
        if os.path.islink(local):
            os.remove(local)
        else:
            shutil.rmtree(local)
    os.symlink(remote, local)
    print(f'  linked: {d} → Drive')

os.makedirs(f'{DRIVE_DIR}/outputs/mocov3_vits_seed42', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
print('\nSetup 완료!')

In [ ]:
# Cell 4 — 학습 시작 (자동 resume + 실시간 로그 출력)
import glob, os, subprocess, sys

WORK_DIR = '/content/ssl_project'
os.chdir(WORK_DIR)

# 마지막 체크포인트 자동 탐색
ckpts = sorted(glob.glob('outputs/mocov3_vits_seed42/ckpt_ep*.pth'))
if ckpts:
    print(f'Resume: {ckpts[-1]}')
    resume_args = ['--resume', ckpts[-1]]
else:
    print('처음부터 학습 시작')
    resume_args = []

cmd = [
    'python3', '-u', 'scripts/train_mocov3.py',
    '--num-workers', '2',
    '--save-every', '5',
] + resume_args

# stdout + stderr 를 셀에 실시간 출력
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n학습 종료 (exit code: {proc.returncode})')

In [ ]:
# Cell 5 — 학습 완료 후 / 세션 재시작 후 상태 확인
# (Cell 4 실행 중에는 실행 불가)
!echo '=== 최근 로그 ==='
!tail -n 20 /content/ssl_project/logs/mocov3_vits_seed42.log
!echo ''
!echo '=== 저장된 체크포인트 ==='
!ls -lh /content/ssl_project/outputs/mocov3_vits_seed42/